# 04 · Comparación y ajuste del clustering

Compara K-Means y Gaussian Mixture para varios números de clústeres. Las métricas orientan la decisión, pero la solución final debe ser interpretable y estable.

In [ ]:
from pathlib import Path
import json

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score, silhouette_score

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
MODELS_DIR = ROOT / 'outputs' / 'models'
REPORTS_DIR = ROOT / 'outputs' / 'reports'
FIGURES_DIR = ROOT / 'outputs' / 'figures'
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

X = np.load(MODELS_DIR / 'matriz_preprocesada.npy')
municipios = pd.read_parquet(MODELS_DIR / 'municipios_modelo.parquet')
configuracion = json.loads((MODELS_DIR / 'configuracion_variables.json').read_text(encoding='utf-8'))
VARIABLES = configuracion['variables']
RANDOM_STATE = 42
print(X.shape, municipios.shape)

## Comparar soluciones

In [ ]:
def evaluar_solucion(X, etiquetas):
    tamanios = pd.Series(etiquetas).value_counts()
    muestra = min(5_000, len(X))
    return {
        'silhouette': silhouette_score(
            X, etiquetas, sample_size=muestra, random_state=RANDOM_STATE
        ),
        'davies_bouldin': davies_bouldin_score(X, etiquetas),
        'calinski_harabasz': calinski_harabasz_score(X, etiquetas),
        'cluster_minimo': int(tamanios.min()),
        'cluster_maximo': int(tamanios.max()),
    }

resultados = []
modelos_candidatos = {}
for k in range(2, 11):
    kmeans = KMeans(n_clusters=k, n_init=30, random_state=RANDOM_STATE)
    etiquetas = kmeans.fit_predict(X)
    resultados.append({'algoritmo': 'kmeans', 'k': k, **evaluar_solucion(X, etiquetas)})
    modelos_candidatos[('kmeans', k)] = kmeans

for k in range(2, 9):
    gmm = GaussianMixture(
        n_components=k, covariance_type='diag', n_init=5, random_state=RANDOM_STATE
    )
    etiquetas = gmm.fit_predict(X)
    resultados.append({
        'algoritmo': 'gmm',
        'k': k,
        'bic': gmm.bic(X),
        'aic': gmm.aic(X),
        **evaluar_solucion(X, etiquetas),
    })
    modelos_candidatos[('gmm', k)] = gmm

resultados = pd.DataFrame(resultados)
resultados.to_csv(REPORTS_DIR / 'metricas_clustering.csv', index=False)
display(resultados.sort_values(['silhouette', 'davies_bouldin'], ascending=[False, True]))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for algoritmo, grupo in resultados.groupby('algoritmo'):
    grupo = grupo.sort_values('k')
    axes[0].plot(grupo['k'], grupo['silhouette'], marker='o', label=algoritmo)
    axes[1].plot(grupo['k'], grupo['davies_bouldin'], marker='o', label=algoritmo)
axes[0].set(title='Silhouette (mayor es mejor)', xlabel='k')
axes[1].set(title='Davies-Bouldin (menor es mejor)', xlabel='k')
for ax in axes:
    ax.grid(alpha=0.25)
    ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'comparacion_clustering.png', dpi=160)
plt.show()

## Elegir solución

La selección automática inferior es únicamente un punto de partida. Cambia `ALGORITMO_FINAL` y `K_FINAL` después de interpretar los perfiles y comprobar que no aparezcan clústeres residuales muy pequeños.

In [ ]:
candidatos_validos = resultados.query('cluster_minimo >= 30')
sugerencia = candidatos_validos.sort_values(
    ['silhouette', 'davies_bouldin'], ascending=[False, True]
).iloc[0]
print('Sugerencia inicial:', sugerencia[['algoritmo', 'k', 'silhouette', 'davies_bouldin']].to_dict())

ALGORITMO_FINAL = str(sugerencia['algoritmo'])
K_FINAL = int(sugerencia['k'])
modelo_final = modelos_candidatos[(ALGORITMO_FINAL, K_FINAL)]
etiquetas = modelo_final.predict(X)

asignaciones = municipios.copy()
asignaciones['cluster'] = etiquetas.astype('int16')
display(asignaciones['cluster'].value_counts().sort_index().to_frame('municipios'))

## Representación PCA y guardado

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
coordenadas = pca.fit_transform(X)

fig, ax = plt.subplots(figsize=(10, 7))
dispersion = ax.scatter(
    coordenadas[:, 0], coordenadas[:, 1], c=etiquetas, s=10, cmap='tab10', alpha=0.65
)
ax.set(
    xlabel=f'CP1 ({pca.explained_variance_ratio_[0]:.1%})',
    ylabel=f'CP2 ({pca.explained_variance_ratio_[1]:.1%})',
    title=f'{ALGORITMO_FINAL.upper()} con {K_FINAL} clústeres',
)
fig.colorbar(dispersion, ax=ax, label='Clúster')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'clusters_pca.png', dpi=160)
plt.show()

joblib.dump(modelo_final, MODELS_DIR / 'modelo_clustering.joblib')
joblib.dump(pca, MODELS_DIR / 'pca_visualizacion.joblib')
asignaciones.to_parquet(MODELS_DIR / 'asignaciones_cluster.parquet', index=False)
seleccion = {'algoritmo': ALGORITMO_FINAL, 'k': K_FINAL, 'random_state': RANDOM_STATE}
(MODELS_DIR / 'configuracion_clustering.json').write_text(
    json.dumps(seleccion, ensure_ascii=False, indent=2), encoding='utf-8'
)